# Partition consistency — does the districting change the pre-drift world?

On D-Town every fast_greedy world runs PU9 / PU11 and leaves PU6 idle in the **init** window, and every
girvan_newman world does the opposite; KY7 shows a smaller version (init duty 0.549 vs 0.556). A partition
only labels nodes, so before any drift two worlds of the same network should be hydraulically identical —
unless some input is keyed by the partition. Until that is explained, every cross-partition comparison on
D-Town mixes the effect of the districting with the effect of a different operating regime.

Every pair of ok worlds of the study is compared on the init window they share (all months before either
world's first drift month), in three layers, from cause to effect:

| layer | question |
|---|---|
| P1 inputs | which resolved parameters differ between the two worlds — and which differ **only between partitions**? |
| P2 demand | is the demand of every node identical over the shared init window? where does it first differ? |
| P3 hydraulics | do pump states and tank exchange agree? where do they first diverge? |

Pairs **within** a partition are the reference: they share the partition, so whatever differs between
them is the drift setup (which district, which map), and must not show up in the init window. A
difference that appears **only between** partitions is keyed by the partition.

In [1]:
%matplotlib inline
import warnings, logging, sys, pathlib, json
from itertools import combinations
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

REPO = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "src" / "fedwater").is_dir()), None)
assert REPO is not None, "run this from inside the fedwater repo"
LAB_DIR = REPO / "notebooks" / "fedwater_lab_refactor"     # adjust to your layout
for p in (REPO / "src", LAB_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
from fedwater.reporting import Atlas

warnings.filterwarnings("ignore")
logging.getLogger("kedro").setLevel(logging.WARNING)
pd.set_option("display.width", 230, "display.max_columns", 60, "display.max_rows", 300,
              "display.max_colwidth", 120)

EXPERIMENTS = None              # None -> <project>/data/09_experiments
STUDY = None                    # None -> the latest study
ATLAS = Atlas(root=EXPERIMENTS)
STUDY = STUDY or ATLAS.studies()[0]
WORLDS = ATLAS.worlds(STUDY, ok_only=True).reset_index(drop=True)
print(f"study {STUDY}: {len(WORLDS)} ok worlds")
display(WORLDS[["sim_hash", "network", "districting", "drift_district"]])

study dtown_districting_all_districts: 10 ok worlds


,sim_hash,network,districting,drift_district
0,0d26d19110e1,dtown,fast_greedy,District_A
1,1ca8610b6a11,dtown,fast_greedy,District_B
2,57c3c8bf631e,dtown,fast_greedy,District_C
3,bac4d73032fa,dtown,fast_greedy,District_D
4,8ce38ac48e7f,dtown,fast_greedy,District_E
5,8190c13b8b41,dtown,girvan_newman,District_A
6,7c8f5a9856af,dtown,girvan_newman,District_B
7,4cc1d3b44da3,dtown,girvan_newman,District_C
8,18c49d14f008,dtown,girvan_newman,District_D
9,583ea08f60b5,dtown,girvan_newman,District_E


### P1 — resolved parameters

Each world's resolved parameters are flattened to `dotted.key → value` (lists and nested lists compared as
JSON) and compared pairwise. A key is **partition-keyed** if it differs in some between-partition pair and
in no within-partition pair. Keys that differ within partitions too (the drifting district, the drift map,
hashes) are listed separately. The network-prep report is compared the same way.

In [2]:
def flatten(d, pre=""):
    out = {}
    if isinstance(d, dict):
        for k, v in d.items():
            out.update(flatten(v, f"{pre}{k}."))
    elif isinstance(d, (list, tuple, set)):
        out[pre[:-1]] = json.dumps(list(d), default=str, sort_keys=True)
    else:
        out[pre[:-1]] = d
    return out


def same(a, b):
    if isinstance(a, float) and isinstance(b, float):
        return (np.isnan(a) and np.isnan(b)) or a == b
    return a == b


VIEWS = {r["sim_hash"]: ATLAS.world(r["sim_hash"]) for _, r in WORLDS.iterrows()}
FLAT = {h: flatten(w.params) for h, w in VIEWS.items()}
PREP = {}
for h, w in VIEWS.items():
    try:
        PREP[h] = flatten(w.load("network_prep_report").iloc[0].to_dict())
    except Exception as exc:
        PREP[h] = {"_unavailable": type(exc).__name__}
META = WORLDS.set_index("sim_hash")

pairs = []
for a, b in combinations(WORLDS["sim_hash"], 2):
    if META.at[a, "network"] != META.at[b, "network"]:
        continue
    kind = "within" if META.at[a, "districting"] == META.at[b, "districting"] else "between"
    keys = sorted(set(FLAT[a]) | set(FLAT[b]))
    dkeys = [k for k in keys if not same(FLAT[a].get(k, "<missing>"), FLAT[b].get(k, "<missing>"))]
    pkeys = [k for k in sorted(set(PREP[a]) | set(PREP[b]))
             if not same(PREP[a].get(k, "<missing>"), PREP[b].get(k, "<missing>"))]
    pairs.append({"a": a, "b": b, "network": META.at[a, "network"], "pair": kind,
                  "part_a": META.at[a, "districting"], "part_b": META.at[b, "districting"],
                  "drift_a": META.at[a, "drift_district"], "drift_b": META.at[b, "drift_district"],
                  "param_keys": dkeys, "prep_keys": pkeys})
PAIRS = pd.DataFrame(pairs)

within = set(k for ks in PAIRS.loc[PAIRS["pair"] == "within", "param_keys"] for k in ks)
between = set(k for ks in PAIRS.loc[PAIRS["pair"] == "between", "param_keys"] for k in ks)
keyed = sorted(between - within)
display(Markdown(f"**Partition-keyed parameters** ({len(keyed)}) — differ between partitions, never within"))
if keyed:
    rep = {}
    for h in WORLDS["sim_hash"]:
        rep[f"{META.at[h, 'districting']} · {META.at[h, 'drift_district']} · {h[:6]}"] = \
            {k: FLAT[h].get(k, "<missing>") for k in keyed}
    display(pd.DataFrame(rep))
display(Markdown(f"**Parameters that differ within partitions too** ({len(within)})"))
print("\n".join(sorted(within)) or "(none)")
display(Markdown("**Network-prep report keys that differ** (per pair kind)"))
display(PAIRS.assign(n_prep=PAIRS["prep_keys"].map(len),
                     prep=PAIRS["prep_keys"].map(lambda k: ", ".join(k)[:150]))
        .groupby("pair")[["n_prep"]].agg(["min", "max"]))
print("prep keys ever differing:",
      sorted(set(k for ks in PAIRS["prep_keys"] for k in ks)) or "(none)")

**Partition-keyed parameters** (1) — differ between partitions, never within

,fast_greedy · District_A · 0d26d1,fast_greedy · District_B · 1ca861,fast_greedy · District_C · 57c3c8,fast_greedy · District_D · bac4d7,fast_greedy · District_E · 8ce38a,girvan_newman · District_A · 8190c1,girvan_newman · District_B · 7c8f5a,girvan_newman · District_C · 4cc1d3,girvan_newman · District_D · 18c49d,girvan_newman · District_E · 583ea0
districting.methods,"[""fast_greedy""]","[""fast_greedy""]","[""fast_greedy""]","[""fast_greedy""]","[""fast_greedy""]","[""girvan_newman""]","[""girvan_newman""]","[""girvan_newman""]","[""girvan_newman""]","[""girvan_newman""]"


**Parameters that differ within partitions too** (3)

scenario.drift.seed_node
scenario.drift.tgt_district
scenario.drift.to_land_use


**Network-prep report keys that differ** (per pair kind)

n_prep    
           min max
pair              
between      0   0
within       0   0

prep keys ever differing: (none)


### P2 / P3 — demand and hydraulics over the shared init window

Per world: the demand of every junction, every pump's on/off state (flow above a relative floor), and
every tank's net inflow (signed sum of its incident link flows), over all months before its first drift
month. Per pair, over the months **both** worlds are still in init:

| column | definition | reading |
|---|---|---|
| `dem_nodes_diff` | share of junctions whose demand differs anywhere in the window (relative floor 1e-9) | 0 = identical demand input |
| `dem_rel_max` | $\max_{t,n}\lvert d^a_{n,t}-d^b_{n,t}\rvert/\overline{\lvert d\rvert}$ | size of the largest node-level difference |
| `tot_rel_rms` | RMS of the total-demand difference / mean total demand | the difference as the network sees it |
| `dem_first_day` | first day with any node-level difference | 0 = different from the first step |
| `pump_mismatch` | share of (step, pump) cells with different on/off state | 0 = identical control trajectory |
| `pump_first_day` | first day any pump's state differs | after `dem_first_day` = the demand difference drives it |
| `tank_rel_rms` | RMS of tank net-inflow differences / mean \|net inflow\| | |

A partition-keyed cause predicts: within-partition pairs all zero; between-partition pairs non-zero in
P2 **and** P3, with `pump_first_day ≥ dem_first_day`. A difference in P3 with none in P2 points instead at
the network or the solver setup (P1's prep keys), or at initial conditions.

In [3]:
def storage_links(wn, cols):
    cols = set(cols)
    pumps = [p for p in wn.pump_name_list if p in cols]
    tanks = {}
    for t in wn.tank_name_list:
        inc = [(ln, 1.0 if wn.get_link(ln).end_node_name == t else -1.0)
               for ln in wn.get_links_for_node(t) if ln in cols]
        if inc:
            tanks[t] = inc
    return pumps, tanks


SNAP = {}
for i, (h, w) in enumerate(VIEWS.items()):
    t = w.params["time"]
    sd = int(round(24.0 / float(t["resolution_h"])))
    sm = int(t["days_per_month"]) * sd
    sch = w.catalog.load("gt_drift_schedule")
    first = int(sch["drift_month"].min()) if sch is not None and len(sch) else int(t["n_months"])
    wn = w.network_model
    dem = w.catalog.load("demand_series")
    nodes = [c for c in dem.columns if c in set(wn.junction_name_list)]
    fl = w.catalog.load("flows")
    pumps, tanks = storage_links(wn, fl.columns)
    n = first * sm
    SNAP[h] = {"sd": sd, "sm": sm, "first": first, "nodes": nodes,
               "dem": dem[nodes].to_numpy(np.float64)[:n],
               "pumps": pumps, "tanks": list(tanks),
               "on": np.column_stack([np.abs(fl[p].to_numpy(float)) >
                                      1e-6 * max(np.abs(fl[p].to_numpy(float)).max(), 1e-12)
                                      for p in pumps])[:n] if pumps else np.zeros((n, 0), bool),
               "tin": np.column_stack([np.sum([s * fl[ln].to_numpy(float) for ln, s in inc], axis=0)
                                       for inc in tanks.values()])[:n] if tanks else np.zeros((n, 0))}
    del dem, fl
    print(f"\r  loaded {i + 1}/{len(VIEWS)}", end="", flush=True)
print()


def first_day(mask_rows, sd):
    idx = np.flatnonzero(mask_rows)
    return float(idx[0] // sd) if len(idx) else np.nan


rows = []
for _, p in PAIRS.iterrows():
    A, B = SNAP[p["a"]], SNAP[p["b"]]
    n = min(len(A["dem"]), len(B["dem"]))
    row = {k: p[k] for k in ("network", "pair", "part_a", "part_b", "drift_a", "drift_b")}
    row.update({"a": p["a"][:8], "b": p["b"][:8], "init_days": n // A["sd"],
                "n_param_keys": len(p["param_keys"]), "n_prep_keys": len(p["prep_keys"])})
    if A["nodes"] == B["nodes"]:
        da, db = A["dem"][:n], B["dem"][:n]
        scale = max(np.abs(da).mean(), 1e-12)
        dif = np.abs(da - db) > 1e-9 * scale
        ta, tb = da.sum(axis=1), db.sum(axis=1)
        row.update({"dem_nodes_diff": float(dif.any(axis=0).mean()),
                    "dem_rel_max": float(np.abs(da - db).max() / scale),
                    "tot_rel_rms": float(np.sqrt(((ta - tb) ** 2).mean()) / max(np.abs(ta).mean(), 1e-12)),
                    "dem_first_day": first_day(dif.any(axis=1), A["sd"])})
    else:
        row["dem_nodes_diff"] = "node sets differ"
    if A["pumps"] == B["pumps"] and A["on"].shape[1]:
        mm = A["on"][:n] != B["on"][:n]
        row.update({"pump_mismatch": float(mm.mean()),
                    "pump_first_day": first_day(mm.any(axis=1), A["sd"]),
                    "pumps_differing": ",".join(pp for pp, m in zip(A["pumps"], mm.any(axis=0)) if m)})
    if A["tanks"] == B["tanks"] and A["tin"].shape[1]:
        ia, ib = A["tin"][:n], B["tin"][:n]
        row["tank_rel_rms"] = float(np.sqrt(((ia - ib) ** 2).mean()) / max(np.abs(ia).mean(), 1e-12))
    rows.append(row)
P23 = pd.DataFrame(rows)

display(Markdown("**Summary by pair kind**"))
num = [c for c in ("dem_nodes_diff", "dem_rel_max", "tot_rel_rms", "dem_first_day",
                   "pump_mismatch", "pump_first_day", "tank_rel_rms") if c in P23]
display(P23.assign(**{c: pd.to_numeric(P23[c], errors="coerce") for c in num})
        .groupby(["network", "pair"])[num].agg(["min", "median", "max"]).round(4).T)
display(Markdown("**Every pair**"))
display(P23.sort_values(["network", "pair", "part_a", "part_b", "drift_a", "drift_b"])
        .round(4).reset_index(drop=True))

  loaded 10/10


**Summary by pair kind**

network                  dtown       
pair                   between within
dem_nodes_diff min      1.0000    0.0
               median   1.0000    0.0
               max      1.0000    0.0
dem_rel_max    min     18.1472    0.0
               median  18.1472    0.0
               max     18.1472    0.0
tot_rel_rms    min      0.0251    0.0
               median   0.0251    0.0
               max      0.0251    0.0
dem_first_day  min      0.0000    NaN
               median   0.0000    NaN
               max      0.0000    NaN
pump_mismatch  min      0.1911    0.0
               median   0.1911    0.0
               max      0.1911    0.0
pump_first_day min      0.0000    NaN
               median   0.0000    NaN
               max      0.0000    NaN
tank_rel_rms   min      1.7963    0.0
               median   1.7963    0.0
               max      1.7963    0.0

**Every pair**

,network,pair,part_a,part_b,drift_a,drift_b,a,b,init_days,n_param_keys,n_prep_keys,dem_nodes_diff,dem_rel_max,tot_rel_rms,dem_first_day,pump_mismatch,pump_first_day,pumps_differing,tank_rel_rms
0,dtown,between,fast_greedy,girvan_newman,District_A,District_A,0d26d191,8190c13b,252,2,0,1.0,18.1472,0.0251,0.0,0.1911,0.0,"PU1,PU2,PU4,PU6,PU7,PU8,PU9,PU10,PU11",1.7963
1,dtown,between,fast_greedy,girvan_newman,District_A,District_B,0d26d191,7c8f5a98,252,3,0,1.0,18.1472,0.0251,0.0,0.1911,0.0,"PU1,PU2,PU4,PU6,PU7,PU8,PU9,PU10,PU11",1.7963
2,dtown,between,fast_greedy,girvan_newman,District_A,District_C,0d26d191,4cc1d3b4,252,4,0,1.0,18.1472,0.0251,0.0,0.1911,0.0,"PU1,PU2,PU4,PU6,PU7,PU8,PU9,PU10,PU11",1.7963
3,dtown,between,fast_greedy,girvan_newman,District_A,District_D,0d26d191,18c49d14,252,3,0,1.0,18.1472,0.0251,0.0,0.1911,0.0,"PU1,PU2,PU4,PU6,PU7,PU8,PU9,PU10,PU11",1.7963
4,dtown,between,fast_greedy,girvan_newman,District_A,District_E,0d26d191,583ea08f,252,3,0,1.0,18.1472,0.0251,0.0,0.1911,0.0,"PU1,PU2,PU4,PU6,PU7,PU8,PU9,PU10,PU11",1.7963
5,dtown,between,fast_greedy,girvan_newman,District_B,District_A,1ca8610b,8190c13b,252,3,0,1.0,18.1472,0.0251,0.0,0.1911,0.0,"PU1,PU2,PU4,PU6,PU7,PU8,PU9,PU10,PU11",1.7963
6,dtown,between,fast_greedy,girvan_newman,District_B,District_B,1ca8610b,7c8f5a98,252,2,0,1.0,18.1472,0.0251,0.0,0.1911,0.0,"PU1,PU2,PU4,PU6,PU7,PU8,PU9,PU10,PU11",1.7963
7,dtown,between,fast_greedy,girvan_newman,District_B,District_C,1ca8610b,4cc1d3b4,252,3,0,1.0,18.1472,0.0251,0.0,0.1911,0.0,"PU1,PU2,PU4,PU6,PU7,PU8,PU9,PU10,PU11",1.7963
8,dtown,between,fast_greedy,girvan_newman,District_B,District_D,1ca8610b,18c49d14,252,4,0,1.0,18.1472,0.0251,0.0,0.1911,0.0,"PU1,PU2,PU4,PU6,PU7,PU8,PU9,PU10,PU11",1.7963
9,dtown,between,fast_greedy,girvan_newman,District_B,District_E,1ca8610b,583ea08f,252,4,0,1.0,18.1472,0.0251,0.0,0.1911,0.0,"PU1,PU2,PU4,PU6,PU7,PU8,PU9,PU10,PU11",1.7963


### Where in the network the demand differs (only if P2 finds a difference)

For the between-partition pairs with a demand difference: which junctions differ, grouped by the district
each partition assigns them to. If the difference follows district membership (e.g. all nodes of one
district in one partition and not in the other), the input is keyed by district — the next place to look
is how the init land-use map is assigned to districts.

In [4]:
def district_of(w):
    dy = w.catalog.load("districts")
    return {str(n): d for d, nodes in dy["districts"].items() for n in nodes}


DIST = {h: district_of(w) for h, w in VIEWS.items()}
out = []
for _, p in PAIRS[PAIRS["pair"] == "between"].iterrows():
    A, B = SNAP[p["a"]], SNAP[p["b"]]
    if A["nodes"] != B["nodes"]:
        continue
    n = min(len(A["dem"]), len(B["dem"]))
    scale = max(np.abs(A["dem"][:n]).mean(), 1e-12)
    diff_nodes = np.abs(A["dem"][:n] - B["dem"][:n]).max(axis=0) > 1e-9 * scale
    if not diff_nodes.any():
        continue
    nd = pd.DataFrame({"node": A["nodes"], "diff": diff_nodes,
                       "dist_a": [DIST[p["a"]].get(nn, "?") for nn in A["nodes"]],
                       "dist_b": [DIST[p["b"]].get(nn, "?") for nn in A["nodes"]],
                       "mean_a": A["dem"][:n].mean(axis=0), "mean_b": B["dem"][:n].mean(axis=0)})
    g = (nd.groupby(["dist_a", "dist_b"])
         .agg(nodes=("node", "size"), differ=("diff", "mean"),
              mean_a=("mean_a", "sum"), mean_b=("mean_b", "sum")).reset_index())
    g.insert(0, "pair", f"{p['part_a']}·{p['drift_a']} vs {p['part_b']}·{p['drift_b']}")
    out.append(g)
if out:
    display(pd.concat(out, ignore_index=True).round(4))
else:
    print("no between-partition pair has a demand difference in the shared init window")

,pair,dist_a,dist_b,nodes,differ,mean_a,mean_b
0,fast_greedy·District_A vs girvan_newman·District_A,District_A,District_B,93,1.0,33.7594,19.8623
1,fast_greedy·District_A vs girvan_newman·District_A,District_A,District_E,2,1.0,0.7288,0.9620
2,fast_greedy·District_A vs girvan_newman·District_A,District_B,District_A,31,1.0,7.5217,12.0133
3,fast_greedy·District_A vs girvan_newman·District_A,District_B,District_D,58,1.0,12.8052,20.4564
4,fast_greedy·District_A vs girvan_newman·District_A,District_C,District_A,3,1.0,0.8987,1.4391
5,fast_greedy·District_A vs girvan_newman·District_A,District_C,District_C,32,1.0,7.1099,6.9005
6,fast_greedy·District_A vs girvan_newman·District_A,District_C,District_E,50,1.0,11.2829,24.2521
7,fast_greedy·District_A vs girvan_newman·District_A,District_D,District_A,36,1.0,10.9758,10.6525
8,fast_greedy·District_A vs girvan_newman·District_A,District_E,District_C,43,1.0,20.5220,8.9936
9,fast_greedy·District_A vs girvan_newman·District_B,District_A,District_B,93,1.0,33.7594,19.8623
